In [15]:
import pandas as pd

splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'test': 'plain_text/test-00000-of-00001.parquet', 'unsupervised': 'plain_text/unsupervised-00000-of-00001.parquet'}
# df = pd.read_parquet("hf://datasets/stanfordnlp/imdb/" + splits["train"], engine="fastparquet")
df_train = pd.read_parquet("Data/Sentiment/" + splits['train'], engine="fastparquet")
df_test = pd.read_parquet("Data/Sentiment/" + splits['test'], engine="fastparquet")

In [16]:
df_train.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [18]:
df_train.shape, df_test.shape

((25000, 2), (25000, 2))

In [20]:
df_test.head()
df = pd.concat([df_train, df_test], axis=0)
df.shape

(50000, 2)

In [ ]:
df[df['label'] == 1].head()

,text,label
12500,Zentropa has much in common with The Third Man...,1
12501,Zentropa is the most original movie I've seen ...,1
12502,Lars Von Trier is never backward in trying out...,1
12503,*Contains spoilers due to me having to describ...,1
12504,That was the first thing that sprang to mind a...,1


In [22]:
import re

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

tokenize(r"hello what are you doing\?\?")

['hello', 'what', 'are', 'you', 'doing']

In [23]:
vocab = {}
word_freq = {}

for text in df['text']:
    tokens = tokenize(text)
    for token in tokens:
        word_freq[token] = word_freq.get(token, 0) + 1

vocab_size = int(1e4)
sorted_words = sorted(word_freq.items(), key=lambda x:x[1], reverse=True)
most_common= sorted_words[:vocab_size]
vocab = {word: idx+2 for idx, (word, _) in enumerate(most_common)}

In [31]:
from sklearn.model_selection import train_test_split

X, X_test, y, y_test = train_test_split(df['text'], df['label'], test_size=0.2, stratify=df['label'])

X.shape, y.shape

((40000,), (40000,))

In [32]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, stratify=y)
X_train.shape, y_train.shape

((36000,), (36000,))

In [33]:
import numpy as np

vocab["<PAD>"] = 0
vocab["<OOV>"] = 1

def encode(text, max_length=200):
    tokens = tokenize(text)
    sequence = []
    
    for token in tokens:
        sequence.append(vocab.get(token, 1))  # OOV index = 1
    
    # Pad / truncate
    if len(sequence) > max_length:
        sequence = sequence[:max_length]
    else:
        sequence += [0] * (max_length - len(sequence))
    
    return sequence

max_length = 200

X_train_encoded = np.array([encode(text, max_length) for text in X_train])
X_val_encoded = np.array([encode(text, max_length) for text in X_val])


In [ ]:
model = tf.keras.Sequential([
    layers.Embedding(input_dim=vocab_size + 2, output_dim=128, input_length=max_length),
    layers.GlobalAveragePooling1D(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
model.fit(
    X_train_encoded,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val_encoded, y_val)
)